# Prompting and RAG, by hand: few-shot prompts, self-consistency, and retrieve-then-generate

A language model can change its behavior **without any weight update**: you change the prompt. You can show it examples (in-context learning), ask it to reason step by step, or paste in text it has never seen (retrieval-augmented generation, RAG). This notebook builds each piece by hand and checks it with numbers.

| Part | What you build | Numbers you reproduce |
|---|---|---|
| 1 | Zero-shot and few-shot prompts; the chat format | exact prompt strings; label-format compliance with and without examples |
| 2 | Structured output: parse a JSON reply | which of 5 replies parse (3 do not) |
| 3 | Chain-of-thought and self-consistency (majority vote) | accuracy 0.40 with 1 sample, rising with more votes |
| 4 | Chunking, TF-IDF, dense (LSA) embeddings, top-$k$ cosine search | chunk boundaries; TF-IDF weights; nearest chunks |
| 5 | Evaluating retrieval: recall@$k$ and precision@$k$ | recall and precision of three retrievers on 10 labeled questions |
| 6 | The RAG prompt and the retrieve-then-generate pipeline | exact prompt; answers with and without context |
| 7 | Failure cases: stale index, missing context, context budget | when RAG answers wrongly, and why |

**How it works.** There are eight small `TODO` functions. Each is followed by a check cell. Until you fill a `TODO`, its check prints a reminder and skips, so the notebook always runs top to bottom.

Everything runs on a CPU in a few minutes. The retrieval parts use only `numpy` and a tiny text corpus written into the notebook, about a made-up town, so no model can know the answers from pretraining. Some cells also call a small open chat model (`Qwen/Qwen2.5-0.5B-Instruct`, about 1 GB) through `transformers`. If it cannot be loaded, or you set `USE_LLM = False`, those cells print a note and skip; every check still runs. A small model on a CPU is slow and makes mistakes, which is fine: we read its behavior, we do not grade it.

In [1]:
import re, json, math, time
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

T_START = time.time()
np.random.seed(0)

def attempt(fn, *args, **kwargs):
    """Run a TODO function; if it is not written yet, say so and return None instead of crashing."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"[TODO] {fn.__name__} is not implemented yet: skipping this check.")
        return None

print("numpy", np.__version__)

numpy 2.3.5


### An optional small chat model

The next cell tries to load a 0.5B-parameter instruction-tuned model. The first run downloads it (about 1 GB). `chat(messages)` turns a list of chat messages into the model's prompt format and generates a reply. Greedy decoding (`temperature=0`) always picks the most likely next token; sampling (`temperature>0`) draws at random, which we need in Part 3.

In [2]:
USE_LLM = True                              # set to False to skip every model call
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
llm, tok = None, None
if USE_LLM:
    try:
        import torch
        import transformers
        from transformers import AutoModelForCausalLM, AutoTokenizer
        transformers.logging.set_verbosity_error()
        torch.manual_seed(0)
        torch.set_num_threads(4)
        tok = AutoTokenizer.from_pretrained(LLM_NAME)
        llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).float().eval()
        print(f"loaded {LLM_NAME}: {sum(p.numel() for p in llm.parameters())/1e6:.0f} M parameters")
    except Exception as e:                  # no internet, no transformers, not enough memory ...
        llm, tok = None, None
        print(f"[no LLM] could not load {LLM_NAME} ({type(e).__name__}); model cells will skip.")
else:
    print("[no LLM] USE_LLM is False; model cells will skip.")

def chat(messages, max_new_tokens=60, temperature=0.0, n=1, seed=0):
    """messages: list of {"role": ..., "content": ...}. Returns a list of n reply strings."""
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt")
    torch.manual_seed(seed)
    kw = dict(do_sample=True, temperature=temperature, top_p=1.0, top_k=0) if temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = llm.generate(**ids, max_new_tokens=max_new_tokens, num_return_sequences=n,
                           pad_token_id=tok.eos_token_id, **kw)
    return [tok.decode(o[ids["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]

loaded Qwen/Qwen2.5-0.5B-Instruct: 494 M parameters


## Part 1. In-context learning: zero-shot and few-shot prompts

**In-context learning** means the model adapts to a task from the prompt alone, at inference time. No gradient step, no weight change. Fine-tuning, in contrast, updates the weights with labeled data.

* A **zero-shot** prompt gives only an instruction and the input.
* A **few-shot** prompt adds a handful of solved examples (input, output) before the new input. The examples show the format and the task.

We use one fixed plain-text layout. Every block is separated by one blank line, and the prompt ends with the cue the model must complete:

```
<instruction>

Input: <example 1 input>
Output: <example 1 output>

Input: <query>
Output:
```

In [3]:
# TODO 1: a zero-shot / few-shot prompt in the layout above.
def build_few_shot_prompt(instruction, examples, query):
    """instruction: str. examples: list of (input, output) pairs; an empty list gives a zero-shot prompt.
    query: str. Return one string: the instruction, then one block "Input: ...\nOutput: ..." per example,
    then "Input: <query>\nOutput:". Join the blocks with a blank line ("\n\n"). No trailing space or newline."""
    raise NotImplementedError

In [4]:
# Check 1: exact strings.
INSTR = "Label the sentiment of the review as POS or NEG."
SHOTS = [("The soup was cold and the waiter was rude.", "NEG"),
         ("Lovely staff, and the bread is amazing.", "POS")]
zs = attempt(build_few_shot_prompt, INSTR, [], "Great view, I will come back.")
if zs is not None:
    fs = build_few_shot_prompt(INSTR, SHOTS, "Great view, I will come back.")
    print(fs)
    assert zs == ("Label the sentiment of the review as POS or NEG.\n\n"
                  "Input: Great view, I will come back.\nOutput:")
    assert fs.count("Input:") == 3 and fs.endswith("Output:") and fs.count("\n\n") == 3
    assert fs.split("\n\n")[1] == "Input: The soup was cold and the waiter was rude.\nOutput: NEG"
    print(f"\nzero-shot prompt: {len(zs.split())} words; few-shot prompt: {len(fs.split())} words")
    print("Check 1 passed.")

[TODO] build_few_shot_prompt is not implemented yet: skipping this check.


### The chat format, and what the examples change

Chat models are trained on **messages** with roles: `system` (standing instructions, a role or persona), `user` and `assistant`. The tokenizer's chat template turns them into one string with special tokens. Few-shot examples can be written as earlier user/assistant turns. The cell below prints the template's output, then (if the model loaded) asks it to label 8 short reviews, zero-shot and few-shot.

Here the instruction is deliberately vague: "Classify this review." Zero-shot, the model must guess what output we want, so it answers in its own words. With two examples, the same frozen model picks up both the task and the exact output format (`POS` / `NEG`) from the prompt alone.

In [5]:
REVIEWS = [("Great view, I will come back.", "POS"), ("Never again. Two hours for a cold pizza.", "NEG"),
           ("The pasta was perfect and fairly priced.", "POS"), ("Dirty tables and a bored cashier.", "NEG"),
           ("Best tacos in town, friendly staff.", "POS"), ("Overpriced and bland.", "NEG"),
           ("A cozy place with a kind owner.", "POS"), ("My order was wrong twice.", "NEG")]

def as_messages(instruction, examples, query, system="You are a helpful assistant."):
    msgs = [{"role": "system", "content": system}]
    for x, y in examples:                        # few-shot examples as earlier turns
        msgs += [{"role": "user", "content": f"{instruction}\n{x}"}, {"role": "assistant", "content": y}]
    msgs.append({"role": "user", "content": f"{instruction}\n{query}"})
    return msgs

VAGUE = "Classify this review."
if llm is not None:
    print(tok.apply_chat_template(as_messages(VAGUE, SHOTS[:1], REVIEWS[0][0]), tokenize=False,
                                  add_generation_prompt=True))
    for name, shots in [("zero-shot", []), ("few-shot (2 examples)", SHOTS)]:
        replies = [chat(as_messages(VAGUE, shots, x), max_new_tokens=8)[0] for x, _ in REVIEWS]
        in_format = sum(x in ("POS", "NEG") for x in replies)
        correct = sum(x == y for x, (_, y) in zip(replies, REVIEWS))
        print(f"{name:22s}: {in_format}/8 replies are exactly POS/NEG, {correct}/8 correct.")
        print(f"{'':22s}  first replies: {replies[:3]}")
else:
    print("[no LLM] skipping the zero-shot vs few-shot comparison.")

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Classify this review.
The soup was cold and the waiter was rude.<|im_end|>
<|im_start|>assistant
NEG<|im_end|>
<|im_start|>user
Classify this review.
Great view, I will come back.<|im_end|>
<|im_start|>assistant



zero-shot             : 0/8 replies are exactly POS/NEG, 0/8 correct.
                        first replies: ['Positive', 'This review can be classified as negative.', 'This review can be classified as positive.']


few-shot (2 examples) : 8/8 replies are exactly POS/NEG, 8/8 correct.
                        first replies: ['POS', 'NEG', 'POS']


## Part 2. Prompt patterns: role, instructions, structured output

Useful patterns from the lecture:

* **Instructions**: say exactly what to do and what to return.
* **Role / system prompt**: standing instructions that hold for the whole conversation.
* **Structured output**: ask for JSON with named fields, so a program can read the answer.

A program must still **check** the reply. Models wrap JSON in prose or code fences, drop fields, or break the syntax. Write a parser that finds the JSON object in a reply and rejects it if it is broken or incomplete.

In [6]:
# TODO 2: read a JSON object out of a model reply.
def parse_json_reply(text, required_keys):
    """Take the substring from the FIRST "{" to the LAST "}" in text and parse it with json.loads.
    Return the dict if it parses and has every key in required_keys; otherwise return None
    (no braces, invalid JSON, not a dict, or a missing key). Hint: catch json.JSONDecodeError."""
    raise NotImplementedError

In [7]:
# Check 2: five replies a model might give. Three of them must be rejected.
KEYS = ["sentiment", "confidence"]
replies = ['{"sentiment": "POS", "confidence": 0.9}',
           'Sure! Here it is:\n```json\n{"sentiment": "NEG", "confidence": 0.7}\n```',
           '{"sentiment": "POS"}',                        # a field is missing
           "{'sentiment': 'POS', 'confidence': 0.8}",     # single quotes: not valid JSON
           'The review is positive.']                     # no JSON at all
r2 = attempt(parse_json_reply, replies[0], KEYS)
if r2 is not None:
    parsed = [parse_json_reply(t, KEYS) for t in replies]
    for t, p in zip(replies, parsed):
        print(f"{'OK  ' if p else 'FAIL'} {t!r:70.70s} -> {p}")
    assert parsed[0] == {"sentiment": "POS", "confidence": 0.9}
    assert parsed[1] == {"sentiment": "NEG", "confidence": 0.7}
    assert parsed[2] is None and parsed[3] is None and parsed[4] is None
    assert parse_json_reply('[1, 2]', []) is None and parse_json_reply('{"a": {"b": 1}}', ["a"]) == {"a": {"b": 1}}
    print("Check 2 passed.")

[TODO] parse_json_reply is not implemented yet: skipping this check.


In [8]:
# The same request to the model: a role in the system prompt, and an exact output format.
SYSTEM = ("You are a strict data-labeling tool. Reply with JSON only, in the form "
          '{"sentiment": "POS" or "NEG", "confidence": a number between 0 and 1}.')
if llm is not None and r2 is not None:
    ok = 0
    for x, y in REVIEWS[:4]:
        reply = chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": x}], max_new_tokens=30)[0]
        p = parse_json_reply(reply, KEYS)
        ok += p is not None
        print(f"{x:42s} -> {reply!r:55.55s} parsed: {p is not None}")
    print(f"{ok}/4 replies parsed")
else:
    print("[no LLM or TODO 2 missing] skipping the structured-output call.")

[no LLM or TODO 2 missing] skipping the structured-output call.


## Part 3. Chain-of-thought and self-consistency

**Chain-of-thought (CoT)** prompting asks the model to write intermediate steps before the final answer ("Let's think step by step."). The extra generated tokens give the model more computation for multi-step problems.

One chain can still go wrong. **Self-consistency** samples several chains (with temperature $> 0$, so they differ), reads the final answer of each, and returns the **most common** answer. Wrong chains tend to disagree with each other; correct chains agree.

`extract_answer` (given) reads the final number of a chain: the number after the last "answer is", else the number in a `\boxed{}`, else the last number in the text. Write the vote.

In [9]:
def extract_answer(chain):
    """Return the final number of a reasoning chain as a string, or None if it has no number."""
    t = chain.lower().replace(",", "")
    m = (re.findall(r"answer is\D{0,12}?(-?\d+(?:\.\d+)?)", t)       # "The answer is 17."
         or re.findall(r"boxed\{(-?\d+(?:\.\d+)?)\}", t)            # LaTeX style: \boxed{17}
         or re.findall(r"-?\d+(?:\.\d+)?", t))                       # otherwise: the last number
    return m[-1] if m else None

print(extract_answer("3 boxes of 4 is 12, plus 5 is 17. So the answer is 17."))
print(extract_answer("So the total is \\boxed{1,250}."))
print(extract_answer("I am not sure."))

17
1250
None


In [10]:
# TODO 3: self-consistency = majority vote over the final answers.
def majority_vote(answers):
    """answers: list of strings, some may be None (no answer found). Ignore the None entries.
    Return (winner, share): the most common answer and the fraction of the non-None answers that chose it.
    On a tie, return the answer that appears FIRST in the list. If every entry is None, return (None, 0.0)."""
    raise NotImplementedError

In [11]:
# Check 3a: small cases.
r3 = attempt(majority_vote, ["17", "12", "17", None])
if r3 is not None:
    assert r3 == ("17", 2 / 3)
    assert majority_vote(["5", "7"]) == ("5", 0.5)              # tie: first one wins
    assert majority_vote(["7", None, "5", "5", "7"]) == ("7", 0.5)
    assert majority_vote([None, None]) == (None, 0.0)
    print("Check 3a passed.")

[TODO] majority_vote is not implemented yet: skipping this check.


### Self-consistency, simulated

We simulate a model on 2,000 word problems. Each sampled chain ends in the **correct** answer with probability 0.40. Otherwise it ends in one of 6 wrong answers (equally likely, 0.10 each), or it gives no final answer (probability 0.00 here; try 0.1). One chain is right 40% of the time. Because the wrong answers are spread out, the correct one usually wins the vote.

In [12]:
def simulate_self_consistency(n_problems, n_samples, p_correct=0.40, n_wrong=6, rng=None):
    """Return the accuracy of majority voting over n_samples simulated chains per problem."""
    p_wrong = (1 - p_correct) / n_wrong
    choices = ["correct"] + [f"wrong{i}" for i in range(n_wrong)]
    probs = [p_correct] + [p_wrong] * n_wrong
    hits = 0
    for _ in range(n_problems):
        answers = list(rng.choice(choices, size=n_samples, p=probs))
        hits += majority_vote(answers)[0] == "correct"
    return hits / n_problems

if r3 is not None:
    rng = np.random.default_rng(0)
    ks = [1, 3, 5, 9, 15, 25]
    acc = {k: simulate_self_consistency(2000, k, rng=rng) for k in ks}
    for k in ks:
        print(f"{k:>3d} sampled chains -> majority-vote accuracy {acc[k]:.3f}")
    assert abs(acc[1] - 0.40) < 0.03
    assert all(acc[a] < acc[b] for a, b in zip(ks, ks[1:]))         # more votes, more accuracy
    assert acc[25] > 0.85
    plt.figure(figsize=(5, 3))
    plt.plot(ks, [acc[k] for k in ks], "o-")
    plt.axhline(0.40, ls="--", c="gray", label="one chain")
    plt.xlabel("sampled chains per problem"); plt.ylabel("accuracy"); plt.title("Self-consistency (simulated)")
    plt.legend(); plt.tight_layout(); plt.show()
    print("Check 3b passed.")
else:
    print("[TODO] the simulation needs majority_vote: skipping.")

[TODO] the simulation needs majority_vote: skipping.


The vote works because the wrong chains **disagree** with each other: the correct answer needs only a plurality, not a majority. If the model's wrong chains all agreed on the **same** wrong answer, voting could not fix it (try `n_wrong=1`). Self-consistency also costs $k$ times the generated tokens.

### The real model: direct answer, chain-of-thought, and a vote

Ask the model one two-step word problem three ways: answer directly, one greedy chain of thought, and a vote over 7 sampled chains. A 0.5B model is weak at arithmetic, so any of the three may fail. Read the sampled answers: do the wrong ones agree with each other?

In [13]:
PROBLEM = ("Tom has 4 times as many stickers as Ann. Ann has 7 stickers. Tom gives Ann 5 stickers. "
           "How many stickers does Tom have now?")         # 4 * 7 - 5 = 23
TRUE = "23"
if llm is not None and r3 is not None:
    direct = chat([{"role": "user", "content": PROBLEM + " Reply with only the number."}], max_new_tokens=8)[0]
    print(f"direct answer: {direct!r}  (true: {TRUE})")
    cot_msg = [{"role": "user", "content": PROBLEM + " Think step by step, in a few short lines. "
                                             "End with: The answer is <number>."}]
    greedy = chat(cot_msg, max_new_tokens=200)[0]
    print(f"\ngreedy chain of thought:\n{greedy}\n-> extracted answer: {extract_answer(greedy)}")
    samples = chat(cot_msg, max_new_tokens=200, temperature=0.8, n=7, seed=1)
    answers = [extract_answer(s) for s in samples]
    print(f"\n7 sampled chains (temperature 0.8) end in: {answers}")
    print(f"self-consistency vote (answer, share): {majority_vote(answers)}")
else:
    print("[no LLM or TODO 3 missing] skipping the chain-of-thought call.")

[no LLM or TODO 3 missing] skipping the chain-of-thought call.


## Part 4. Retrieval: chunking, TF-IDF, dense embeddings, top-$k$ search

RAG runs in two phases.

* **Indexing time** (once): split each document into chunks, turn each chunk into a vector, and store the vectors in an index (a **vector database**).
* **Query time** (per question): turn the question into a vector with the **same** encoder, find the $k$ nearest chunk vectors, and paste those chunks into the prompt.

Our corpus is a short FAQ about the made-up town of Alderbrook. Each document has an id, a date and a text. Note `bike-2024` and `bike-2026`: the price changed.

In [14]:
DOCS = [
 ("library", "2025-09", "The Alderbrook Public Library is open 9 am to 8 pm Monday to Friday and 10 am to 4 pm on "
  "Saturday. It is closed on Sunday. A library card is free for residents. You can borrow up to 12 books at a "
  "time for three weeks. The late fee is 25 cents per book per day. The library also lends laptops and "
  "museum passes."),
 ("bike-2024", "2024-03", "The Alderbrook bike share has 15 docking stations in town. An annual bike pass costs "
  "40 dollars. A single ride costs 2 dollars for the first 30 minutes. Bicycles must be returned to any "
  "station before midnight."),
 ("recycling", "2025-05", "Recycling is picked up every other Tuesday. Put paper, cans and plastic bottles in the "
  "blue bin. Glass bottles and jars are not collected at the curb. Take glass to the drop-off center on Mill "
  "Road, open Saturday 8 am to noon. Old electronics also go to the drop-off center."),
 ("clinic", "2025-10", "The Alderbrook Community Clinic takes walk-in patients from 8 to 11 am on weekdays. Flu "
  "shots are free for everyone in October and November. Other vaccines need an appointment. Call the clinic "
  "the day before to book a vaccine appointment."),
 ("parking", "2025-01", "Downtown parking meters cost 1 dollar 50 cents per hour. Parking is free after 6 pm and "
  "all day on Sunday. Saturday parking is not free. Residents can buy a yearly parking permit for 30 dollars at "
  "town hall."),
 ("pool", "2025-04", "The Riverside swimming pool is open from June 1 to Labor Day. Lap swim is from 6 to 8 am "
  "every day. A day pass costs 4 dollars for adults. Children under 12 swim for free. Swim lessons for kids "
  "start in July."),
 ("snow", "2024-11", "When the town declares a snow emergency, do not park on streets marked with blue snow "
  "route signs. Cars left on a snow route are towed after 2 hours. The emergency ends when the plows have "
  "cleared every route."),
 ("trash", "2025-05", "Trash is picked up every Monday. Put the trash cart at the curb by 7 am. Large items "
  "such as sofas and mattresses are picked up on the first Monday of each month if you call town hall "
  "first."),
]
for doc_id, date, text in DOCS:
    print(f"{doc_id:10s} {date}  {len(text.split()):3d} words")

library    2025-09   65 words
bike-2024  2024-03   37 words
recycling  2025-05   50 words
clinic     2025-10   40 words
parking    2025-01   39 words
pool       2025-04   43 words
snow       2024-11   39 words
trash      2025-05   39 words


### Chunking

A chunk should be small enough to be about one thing, and large enough to hold a whole fact. A common scheme is a **sliding window** over words: chunks of `size` words, where each chunk starts `size - overlap` words after the previous one. The overlap keeps a fact that straddles a boundary whole in at least one chunk.

In [15]:
# TODO 4: sliding-window chunking over words.
def chunk_words(text, size, overlap):
    """Split text on whitespace into words. Return a list of chunks (strings, words joined by one space).
    Chunk i starts at word i * (size - overlap) and holds up to `size` words.
    Stop after the first chunk that reaches the last word. Assume 0 <= overlap < size."""
    raise NotImplementedError

In [16]:
# Check 4: a 23-word text, size 10, overlap 3 -> chunks start at words 0, 7, 14.
toy = " ".join(f"w{i}" for i in range(23))
c = attempt(chunk_words, toy, 10, 3)
if c is not None:
    for x in c:
        print(x)
    assert c == [" ".join(f"w{i}" for i in range(0, 10)), " ".join(f"w{i}" for i in range(7, 17)),
                 " ".join(f"w{i}" for i in range(14, 23))]
    assert chunk_words(toy, 10, 0)[-1] == "w20 w21 w22" and len(chunk_words(toy, 10, 0)) == 3
    assert chunk_words("a b c", 10, 2) == ["a b c"]                  # short text: one chunk
    assert len(chunk_words(" ".join(["x"] * 20), 10, 0)) == 2        # exact fit: no empty chunk

    CHUNK_SIZE, OVERLAP = 30, 8
    CHUNKS, CHUNK_DOC = [], []                                        # chunk text, and the doc it came from
    for doc_id, date, text in DOCS:
        for piece in chunk_words(text, CHUNK_SIZE, OVERLAP):
            CHUNKS.append(piece); CHUNK_DOC.append(doc_id)
    print(f"\n{len(DOCS)} documents -> {len(CHUNKS)} chunks of up to {CHUNK_SIZE} words (overlap {OVERLAP})")
    assert len(CHUNKS) == 17
    print("Check 4 passed.")
else:
    CHUNKS = CHUNK_DOC = None

[TODO] chunk_words is not implemented yet: skipping this check.


### Sparse vectors: TF-IDF

Tokenize (lowercase, keep words, drop very common words, strip a plural "s"), then weight each word in a chunk by

$$\text{tf-idf}(w, c) = \text{count}(w, c) \times \text{idf}(w), \qquad \text{idf}(w) = \ln\frac{1 + N}{1 + \text{df}(w)} + 1,$$

where $N$ is the number of chunks and $\text{df}(w)$ is the number of chunks that contain $w$. A word that appears everywhere gets a low weight. Each row is then scaled to unit length, so a dot product of two rows is their **cosine similarity**. Most entries are zero: the vector is **sparse**, and it only matches words that literally appear in both texts.

In [17]:
STOP = set("a an the is are be to of and or in on at for from by with it its this that do does can you i "
           "my me what when where how much many which who any all each per there their your get am pm".split())

def tokenize(text):
    words = re.findall(r"[a-z0-9]+", text.lower())
    words = [w[:-2] if w.endswith("sses") else w for w in words]                      # mattresses -> mattress
    words = [w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w for w in words]  # bottles -> bottle
    return [w for w in words if w not in STOP]

def build_vocab(token_lists):
    return {w: i for i, w in enumerate(sorted({w for toks in token_lists for w in toks}))}

print(tokenize("Where do I take glass bottles on Saturdays?"))

['take', 'glass', 'bottle', 'saturday']


In [18]:
# TODO 5: TF-IDF vectors with unit-length rows.
def tfidf(token_lists, vocab, idf=None):
    """token_lists: list of token lists. vocab: dict word -> column. Ignore tokens not in vocab.
    If idf is None, compute it from token_lists: idf[j] = ln((1 + N) / (1 + df_j)) + 1, N = len(token_lists),
    df_j = number of lists that contain word j at least once. Otherwise use the idf you are given (query time).
    Return (X, idf): X is a float array (len(token_lists), len(vocab)) of count * idf, each row divided by its
    L2 norm (leave all-zero rows as zeros). idf is a float array of length len(vocab)."""
    raise NotImplementedError

In [19]:
# Check 5: three tiny "chunks" with known weights.
toy_docs = [["cat", "sat"], ["cat", "cat", "dog"], ["dog", "ran"]]
toy_vocab = build_vocab(toy_docs)                          # cat 0, dog 1, ran 2, sat 3
X_TFIDF = None
r5 = attempt(tfidf, toy_docs, toy_vocab)
if r5 is not None:
    X, idf = r5
    print("idf:", np.round(idf, 4))
    print("X:\n", np.round(X, 4))
    assert np.allclose(idf, [1 + np.log(4 / 3), 1 + np.log(4 / 3), 1 + np.log(2), 1 + np.log(2)])
    assert np.allclose(X[1], [2 / np.sqrt(5), 1 / np.sqrt(5), 0, 0])     # equal idf: counts 2 and 1
    assert np.allclose(np.linalg.norm(X, axis=1), 1)
    Xq, _ = tfidf([["dog", "zebra"], ["zebra"]], toy_vocab, idf)          # query time: reuse idf
    assert np.allclose(Xq[0], [0, 1, 0, 0]) and np.allclose(Xq[1], 0)

    if CHUNKS is None:
        print("[TODO] the town index also needs chunk_words: skipping it.")
    else:
        CHUNK_TOKS = [tokenize(c) for c in CHUNKS]
        VOCAB = build_vocab(CHUNK_TOKS)
        X_TFIDF, IDF = tfidf(CHUNK_TOKS, VOCAB)
        print(f"\nindex: {X_TFIDF.shape[0]} chunks x {X_TFIDF.shape[1]} words; "
              f"{(X_TFIDF > 0).mean():.1%} of the entries are non-zero")
    print("Check 5 passed.")

[TODO] tfidf is not implemented yet: skipping this check.


### Top-$k$ nearest neighbors by cosine

With unit-length rows, cosine similarity is a dot product: `scores = X @ q`. A vector database does exactly this search (with clever indexes to make it fast for millions of vectors). For ties, prefer the lower index, so results are reproducible.

In [20]:
# TODO 6: top-k search by cosine similarity.
def top_k(query_vec, index, k):
    """query_vec: (d,). index: (n, d) with unit-length (or zero) rows. Return (ids, scores) as numpy arrays
    of length min(k, n): the rows with the largest cosine similarity, best first; ties go to the lower row id.
    Scale query_vec to unit length first (a zero query gives all-zero scores).
    Hint: np.argsort(-scores, kind="stable")."""
    raise NotImplementedError

In [21]:
# Check 6: a toy index, then the town index.
toy_index = np.array([[1, 0], [0.6, 0.8], [0, 1], [0.6, 0.8]])
SEARCH_OK = False
r6 = attempt(top_k, np.array([3.0, 4.0]), toy_index, 3)
if r6 is not None:
    ids, sc = r6
    assert list(ids) == [1, 3, 2] and np.allclose(sc, [1.0, 1.0, 0.8])
    assert list(top_k(np.array([1.0, 0.0]), toy_index, 10)[0]) == [0, 1, 3, 2]
    print("Check 6 passed (toy index).")
if r6 is not None and X_TFIDF is not None:
    SEARCH_OK = True

    def search_tfidf(question, k=3):
        q, _ = tfidf([tokenize(question)], VOCAB, IDF)
        return top_k(q[0], X_TFIDF, k)

    question = "Where do I take glass bottles?"
    ids, sc = search_tfidf(question)
    print(question)
    for i, s in zip(ids, sc):
        print(f"  {s:.3f}  [{CHUNK_DOC[i]}] {CHUNKS[i][:80]}...")
    assert CHUNK_DOC[ids[0]] == "recycling"
elif r6 is not None:
    print("[TODO] searching the town index needs TODOs 4 and 5: skipping.")

[TODO] top_k is not implemented yet: skipping this check.


### Dense vectors: an embedding from co-occurrence (LSA)

TF-IDF fails when the question uses a word the right chunk does not contain. An **embedding** is a short dense vector in which related texts land close together, even with no word in common. Real RAG systems use a trained neural encoder. With no model to download, we use the classic **latent semantic analysis (LSA)**: a truncated SVD $X \approx U_r S_r V_r^\top$ of the TF-IDF matrix. A chunk's embedding is its row of $U_r S_r$; a question's is $q V_r$. Words that occur in the same chunks get mixed into the same few directions.

On a corpus this small, LSA is a crude embedding. Part 5 measures how it does.

In [22]:
def fit_lsa(X, dim):
    U, S, Vt = np.linalg.svd(X, full_matrices=False)
    V = Vt[:dim].T                                           # (vocab, dim)
    E = X @ V                                                # = U_r S_r
    return E / np.linalg.norm(E, axis=1, keepdims=True), V

if SEARCH_OK:
    LSA_DIM = 8
    E_DENSE, V_LSA = fit_lsa(X_TFIDF, LSA_DIM)
    print(f"dense index: {E_DENSE.shape[0]} chunks x {E_DENSE.shape[1]} numbers (vs {X_TFIDF.shape[1]} sparse)")

    def search_dense(question, k=3):
        q, _ = tfidf([tokenize(question)], VOCAB, IDF)
        return top_k(q[0] @ V_LSA, E_DENSE, k)

    def search_hybrid(question, k=3, w=0.5):
        """Mix the two scores: w * dense + (1 - w) * tf-idf, for every chunk."""
        q, _ = tfidf([tokenize(question)], VOCAB, IDF)
        qd = q[0] @ V_LSA
        s = w * (E_DENSE @ (qd / max(np.linalg.norm(qd), 1e-12))) + (1 - w) * (X_TFIDF @ q[0])
        ids = np.argsort(-s, kind="stable")[:k]
        return ids, s[ids]

    question = "How much is the yearly pass for the town bicycles?"
    print(question)
    for name, fn in [("tf-idf", search_tfidf), ("dense ", search_dense)]:
        ids, sc = fn(question)
        print(f"{name}: " + ", ".join(f"{CHUNK_DOC[i]} ({s:.2f})" for i, s in zip(ids, sc)))
else:
    print("[TODO] the dense index needs TODOs 4-6: skipping.")

[TODO] the dense index needs TODOs 4-6: skipping.


## Part 5. Is the retriever any good? recall@$k$ and precision@$k$

If retrieval misses the right chunk, the generator never sees the fact. So measure retrieval on its own, with a small labeled set: each question lists the documents that answer it. For one question, with the top-$k$ retrieved chunks mapped to their documents:

$$\text{recall@}k = \frac{|\text{relevant docs found in the top } k|}{|\text{relevant docs}|}, \qquad
\text{precision@}k = \frac{|\text{top-}k \text{ chunks whose doc is relevant}|}{k}.$$

Average each over the questions. Recall asks "did we find it?"; precision asks "how much of the context is useful?" (the rest uses up the context window).

In [23]:
# TODO 7: recall@k and precision@k for one question.
def recall_precision_at_k(retrieved_docs, relevant_docs, k):
    """retrieved_docs: list of doc ids, best first (one per retrieved chunk, may repeat).
    relevant_docs: non-empty set of doc ids. Use only the first k entries of retrieved_docs.
    Return (recall, precision) with recall = |relevant docs that appear in the top k| / |relevant_docs|
    and precision = (number of the top-k entries that are relevant) / k."""
    raise NotImplementedError

In [24]:
# Check 7: hand-worked cases, then three retrievers on 10 labeled questions.
r7 = attempt(recall_precision_at_k, ["pool", "library", "pool"], {"pool"}, 3)
if r7 is not None:
    assert r7 == (1.0, 2 / 3)
    assert recall_precision_at_k(["pool", "library", "pool"], {"pool"}, 1) == (1.0, 1.0)
    assert recall_precision_at_k(["trash", "pool"], {"pool", "clinic"}, 2) == (0.5, 0.5)
    assert recall_precision_at_k(["trash", "pool"], {"clinic"}, 2) == (0.0, 0.0)
    print("hand-worked cases passed.")

LABELED = [
    ("What time does the library close on Saturday?", {"library"}),
    ("Is there a fee if I return a book late?", {"library"}),
    ("How much is the yearly pass for the town bicycles?", {"bike-2024"}),
    ("Where do I take glass bottles?", {"recycling"}),
    ("Can I get a flu shot without paying?", {"clinic"}),
    ("Is parking free on weekends?", {"parking"}),
    ("When can I swim laps?", {"pool"}),
    ("Where can I leave my car during a snow emergency?", {"snow"}),
    ("How do I get rid of an old mattress?", {"trash"}),
    ("Which days are bins collected?", {"recycling", "trash"}),
]
if r7 is not None and SEARCH_OK:
    results = {}
    print(f"\n{'retriever':10s} {'recall@1':>9s} {'recall@3':>9s} {'precision@3':>12s}")
    for name, fn in [("tf-idf", search_tfidf), ("dense", search_dense), ("hybrid", search_hybrid)]:
        rec1, rec3, prec3 = [], [], []
        for q, rel in LABELED:
            docs = [CHUNK_DOC[i] for i in fn(q, k=3)[0]]
            rec1.append(recall_precision_at_k(docs, rel, 1)[0])
            r3, p3 = recall_precision_at_k(docs, rel, 3)
            rec3.append(r3); prec3.append(p3)
        results[name] = (np.mean(rec1), np.mean(rec3), np.mean(prec3))
        print(f"{name:10s} {results[name][0]:9.2f} {results[name][1]:9.2f} {results[name][2]:12.2f}")
    assert np.allclose(results["tf-idf"], (0.85, 0.95, 20 / 30))
    assert np.allclose(results["dense"], (0.95, 0.95, 22 / 30)) and np.allclose(results["hybrid"], results["dense"])
    print("Check 7 passed.")

[TODO] recall_precision_at_k is not implemented yet: skipping this check.


Read the table. Going from $k=1$ to $k=3$ raises recall but lowers precision: the extra chunks are often off-topic, and they still fill the prompt. TF-IDF puts the wrong chunk first for "yearly pass for the town bicycles" (the parking page says "yearly"); the dense embedding has learned that "pass" goes with the bike page. No retriever gets both documents for "Which days are bins collected?" in its top 3. Which retriever wins depends on the corpus and the encoder, so measure it on a labeled set like this one for your own data.

## Part 6. Retrieve, then generate

Now put the retrieved chunks into the prompt. A common layout:

1. fixed instructions first (answer only from the context; say "I don't know" if the answer is not there),
2. the numbered context chunks, best first,
3. the question **last**, right before the model answers.

The context window is finite, so we add chunks only while they fit in a word budget. A chunk that does not fit is skipped, and later (lower-ranked) chunks are still tried.

In [25]:
RAG_INSTRUCTIONS = ("Answer the question using only the context below. "
                    "If the context does not contain the answer, say \"I don't know\".")

In [26]:
# TODO 8: the RAG prompt with a context budget.

def build_rag_prompt(question, chunks, max_context_words):
    """chunks: list of strings, best first. Go through them in order and keep a chunk if the total number of
    words (len(chunk.split())) of the kept chunks stays <= max_context_words; otherwise skip it and go on.
    Number the kept chunks [1], [2], ... Return the string
      RAG_INSTRUCTIONS + "\n\nContext:\n" + "[1] chunk\n[2] chunk ..." + "\n\nQuestion: " + question + "\nAnswer:"
    with the kept chunks joined by "\n". If no chunk is kept, the context part is "(none)"."""
    raise NotImplementedError

In [27]:
# Check 8: exact prompt strings.
p8 = attempt(build_rag_prompt, "Q?", ["one two three", "four five six seven", "eight"], 4)
if p8 is not None:
    print(p8)
    assert p8 == RAG_INSTRUCTIONS + "\n\nContext:\n[1] one two three\n[2] eight\n\nQuestion: Q?\nAnswer:"
    assert "[1] one two three\n[2] four five six seven\n[3] eight" in build_rag_prompt("Q?", ["one two three", "four five six seven", "eight"], 8)
    assert "Context:\n(none)\n" in build_rag_prompt("Q?", ["too long for it"], 2)
    assert build_rag_prompt("Q?", [], 100).endswith("Question: Q?\nAnswer:")
    print("Check 8 passed.")

[TODO] build_rag_prompt is not implemented yet: skipping this check.


### The full pipeline

`rag_answer` retrieves the top chunks, builds the prompt and generates. With the model, the generator is the LLM. Without it, a tiny **extractive reader** stands in: it returns the context sentence that shares the most words with the question (ties go to the earlier, higher-ranked sentence). It cannot paraphrase or combine two sentences (see its answer to the glass question), but it shows the pipeline end to end, and it can only say what is in the context.

We also ask the model the same questions **without** context. It has never seen Alderbrook, so any specific answer it gives is a guess: a hallucination.

In [28]:
def extractive_reader(question, chunks):
    q = set(tokenize(question))
    best, best_score = "I don't know.", 0
    for ch in chunks:
        for sent in re.split(r"(?<=\.)\s+", ch):
            score = len(q & set(tokenize(sent)))
            if score > best_score:
                best, best_score = sent, score
    return best

def rag_answer(question, k=3, max_context_words=120, search=None, use_llm=True):
    search = search or search_hybrid
    ids, scores = search(question, k=k)
    chunks = [CHUNKS[i] for i in ids]
    if use_llm and llm is not None:
        prompt = build_rag_prompt(question, chunks, max_context_words)
        answer = chat([{"role": "user", "content": prompt}], max_new_tokens=40)[0]
    else:
        answer = extractive_reader(question, chunks)        # the reader sees the same top-k chunks
    return answer, [CHUNK_DOC[i] for i in ids], scores

READY = p8 is not None and SEARCH_OK
DEMO = ["What time does the library close on Saturday?", "Where do I take glass bottles?",
        "How much does a day pass at the pool cost for adults?"]
if READY:
    for q in DEMO:
        ans, docs, _ = rag_answer(q, use_llm=False)
        print(f"Q: {q}\n   retrieved {docs}\n   reader : {ans}")
        if llm is not None:
            print(f"   RAG LLM: {rag_answer(q)[0]}")
            print(f"   no RAG : {chat([{'role': 'user', 'content': q + ' (in the town of Alderbrook)'}], max_new_tokens=40)[0]}")
    assert extractive_reader(DEMO[0], [CHUNKS[i] for i in search_hybrid(DEMO[0])[0]]).startswith(
        "The Alderbrook Public Library is open 9 am to 8 pm")
    assert "4 dollars for adults" in rag_answer(DEMO[2], use_llm=False)[0]
else:
    print("[TODO] the pipeline needs TODOs 4, 5, 6 and 8: skipping.")

[TODO] the pipeline needs TODOs 4, 5, 6 and 8: skipping.


## Part 7. Why RAG helps, and where it fails

RAG grounds the answer in retrieved text, and you can update the corpus without retraining. But the answer can only be as good as what is retrieved and what fits in the prompt. Three failure cases:

### (a) Stale knowledge: the index is out of date

The bike pass price changed in 2026. Our index only has the 2024 page, so the pipeline confidently gives the old price. The fix is not a new model: add the new document to the corpus and re-index (and drop or down-weight the outdated page).

In [29]:
Q_BIKE = "How much does an annual bike pass cost?"
NEW_DOC = ("bike-2026", "2026-01", "Starting in January 2026, the annual bike pass costs 55 dollars. "
           "The single ride price is unchanged. Ten electric bikes were added to the bike share.")
if READY:
    old_answer, old_docs, _ = rag_answer(Q_BIKE, use_llm=False)
    print(f"index without the 2026 page: {old_answer}   (from {old_docs[0]})")
    assert "40 dollars" in old_answer

    # Re-index with the new page, and drop the outdated one.
    DOCS_NEW = [d for d in DOCS if d[0] != "bike-2024"] + [NEW_DOC]
    CHUNKS, CHUNK_DOC = [], []
    for doc_id, date, text in DOCS_NEW:
        for piece in chunk_words(text, CHUNK_SIZE, OVERLAP):
            CHUNKS.append(piece); CHUNK_DOC.append(doc_id)
    CHUNK_TOKS = [tokenize(c) for c in CHUNKS]
    VOCAB = build_vocab(CHUNK_TOKS)
    X_TFIDF, IDF = tfidf(CHUNK_TOKS, VOCAB)
    E_DENSE, V_LSA = fit_lsa(X_TFIDF, LSA_DIM)
    new_answer, new_docs, _ = rag_answer(Q_BIKE, use_llm=False)
    print(f"index with the 2026 page   : {new_answer}   (from {new_docs[0]})")
    assert "55 dollars" in new_answer
    if llm is not None:
        print(f"RAG LLM, updated index     : {rag_answer(Q_BIKE)[0]}")
        print(f"LLM with no context        : {chat([{'role': 'user', 'content': Q_BIKE + ' (Alderbrook bike share)'}], max_new_tokens=40)[0]}")
else:
    print("[TODO] this cell needs the pipeline (TODOs 4, 5, 6 and 8): skipping.")

[TODO] this cell needs the pipeline (TODOs 4, 5, 6 and 8): skipping.


### (b) Missing context: the corpus does not have the answer

Retrieval always returns $k$ chunks, even when none is relevant. A model that is told to use them may still produce an answer from unrelated text. A simple guard: if the best similarity is below a threshold, answer "I don't know" without calling the generator. Compare the top scores of answerable questions with questions the corpus cannot answer. Note that "town" alone is enough to give an unrelated chunk a fair score, so a threshold is a trade-off: set it high enough to refuse all three, and you also refuse a question the corpus can answer.

In [30]:
UNANSWERABLE = ["What is the mayor's salary?", "Who won the town chess tournament?",
                "How tall is the town clock tower?"]
if READY:
    ans_top = [search_tfidf(q, k=1)[1][0] for q, _ in LABELED]
    una_top = [search_tfidf(q, k=1)[1][0] for q in UNANSWERABLE]
    print("best tf-idf score, answerable  :", np.round(ans_top, 2))
    print("best tf-idf score, unanswerable:", np.round(una_top, 2))
    for THRESH in [0.15, 0.26]:
        kept = sum(s >= THRESH for s in ans_top)
        refused = sum(s < THRESH for s in una_top)
        print(f"threshold {THRESH}: {kept}/{len(ans_top)} answerable questions kept, "
              f"{refused}/{len(una_top)} unanswerable refused")
    assert [sum(s >= t for s in ans_top) for t in (0.15, 0.26)] == [10, 9]
    assert [sum(s < t for s in una_top) for t in (0.15, 0.26)] == [1, 3]
    q = UNANSWERABLE[1]
    ans, docs, _ = rag_answer(q, use_llm=False)
    print(f"\nwithout a guard, the reader answers {q!r} with: {ans!r} (retrieved {docs})")
    if llm is not None:
        print(f"RAG LLM: {rag_answer(q)[0]}")
else:
    print("[TODO] this cell needs the pipeline (TODOs 4, 5, 6 and 8): skipping.")

[TODO] this cell needs the pipeline (TODOs 4, 5, 6 and 8): skipping.


### (c) The context budget

A prompt has room for only so many words. If the budget is smaller than the retrieved chunks, the lower-ranked ones are dropped, and so is any fact they held. The cell retrieves the top 5 chunks (hybrid search, updated index) and counts, over the labeled questions, how often a relevant chunk still makes it into the prompt as the budget shrinks. A flat stretch means the relevant chunk for some question is ranked low: it fits only once the budget also holds every chunk ranked above it. Ranking quality and budget work together.

In [31]:
if READY:
    LABELED_NEW = [(q, {"bike-2026"} if rel == {"bike-2024"} else rel) for q, rel in LABELED]
    budgets = [10, 20, 30, 45, 60, 90, 120]
    in_prompt = []
    for B in budgets:
        hits = 0
        for q, rel in LABELED_NEW:
            ids, _ = search_hybrid(q, k=5)
            prompt = build_rag_prompt(q, [CHUNKS[i] for i in ids], B)
            hits += any(CHUNKS[i] in prompt and CHUNK_DOC[i] in rel for i in ids)
        in_prompt.append(hits / len(LABELED_NEW))
        print(f"budget {B:>3d} words: a relevant chunk is in the prompt for {in_prompt[-1]:.0%} of questions")
    assert in_prompt == [0.0, 0.4, 0.9, 0.9, 0.9, 0.9, 1.0]
    plt.figure(figsize=(5, 3))
    plt.plot(budgets, in_prompt, "o-")
    plt.xlabel("context budget (words)"); plt.ylabel("questions with a relevant chunk")
    plt.title("Context budget vs. coverage"); plt.ylim(0, 1.05); plt.tight_layout(); plt.show()
else:
    print("[TODO] this cell needs the pipeline (TODOs 4, 5, 6 and 8): skipping.")

[TODO] this cell needs the pipeline (TODOs 4, 5, 6 and 8): skipping.


## Summary

| Part | Idea | What this notebook showed |
|---|---|---|
| 1 | In-context learning | the same frozen model follows a new output format once the prompt shows two examples |
| 2 | Structured output | ask for JSON, then parse and validate it: replies can be wrapped, incomplete or invalid |
| 3 | Chain-of-thought, self-consistency | a majority vote over sampled chains lifts accuracy well above one chain when wrong answers disagree |
| 4 | Retrieval | chunk with overlap; TF-IDF matches words, a dense embedding compresses co-occurrence; top-$k$ by cosine |
| 5 | Evaluation | recall@$k$ rises with $k$ while precision@$k$ falls; measure retrievers on labeled questions |
| 6 | Retrieve-then-generate | instructions, then context, then the question; the answer comes from the retrieved text |
| 7 | Limits | a stale index gives stale answers; retrieval returns something even when nothing is relevant; a small budget drops facts |

RAG reduces hallucination by giving the model real source text, and it handles new facts by updating the corpus instead of retraining. It does not fix bad retrieval: measure retrieval first.

**Credits.** Few-shot in-context learning: Brown et al., "Language Models are Few-Shot Learners" (NeurIPS 2020). Chain-of-thought: Wei et al. (NeurIPS 2022); "Let's think step by step": Kojima et al. (NeurIPS 2022). Self-consistency: Wang et al. (ICLR 2023). RAG: Lewis et al., "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (NeurIPS 2020). TF-IDF: Sparck Jones (1972); LSA: Deerwester et al. (1990). The model is Qwen2.5-0.5B-Instruct (Qwen team, 2024).

In [32]:
print(f"Total notebook runtime: {time.time() - T_START:.0f} s")

Total notebook runtime: 6 s
